# Industrial Predictive Maintenance

## Notebook 01: Data Ingestion and Profiling

### Objective
This notebook aims to load, understand, and profile
the raw machine degradation dataset before data
cleaning and SQL migration.

### Dataset Overview
The dataset contains three related CSV tables:

- readings.csv: Machine sensor measurements.
- units.csv: Machine metadata.
- ground_truth.csv: Machine degradation measurements.

The dataset is synthetic and is designed to simulate
industrial machine degradation.

In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

print("Environment is ready!")

Environment is ready!


## 2. Data Ingestion

### 2.1 Define Dataset Path

In [6]:
DATA_PATH = Path("../data/raw")

print("Dataset folder exists:", DATA_PATH.exists())

Dataset folder exists: True


### 2.2 Load Raw Datasets

Load the three CSV files without modifying
the original data.

In [7]:
readings = pd.read_csv(DATA_PATH / "readings.csv")
units = pd.read_csv(DATA_PATH / "units.csv")
ground_truth = pd.read_csv(DATA_PATH / "ground_truth.csv")

print("All datasets loaded successfully!")

All datasets loaded successfully!


### 2.3 Dataset Dimensions

Inspect the number of rows and columns
in each dataset.

In [8]:
dataset_summary = pd.DataFrame({
    "Dataset": ["readings", "units", "ground_truth"],
    "Rows": [
        readings.shape[0],
        units.shape[0],
        ground_truth.shape[0]
    ],
    "Columns": [
        readings.shape[1],
        units.shape[1],
        ground_truth.shape[1]
    ]
})

display(dataset_summary)

,Dataset,Rows,Columns
0,readings,23118,13
1,units,100,4
2,ground_truth,23118,3


### 2.4 Dataset Preview

Display the first five records of each table
to understand its initial structure.

In [9]:
print("Readings Dataset:")
display(readings.head())

print("Units Dataset:")
display(units.head())

print("Ground Truth Dataset:")
display(ground_truth.head())

Readings Dataset:


,unit_id,cycle,rul_cycles,machine_failure,failure_mode,tool_wear_min,vibration_mm_s,torque_nm,process_temperature_k,air_temperature_k,rotational_speed_rpm,control_type,split
0,U0001,1,275,0,none,-1.0,0.864,NaN,310.32,299.88,1573.0,B,train
1,U0001,2,274,0,none,0.0,0.810,38.89,310.94,NaN,1592.0,C,train
2,U0001,3,273,0,none,1.0,0.850,NaN,310.57,300.28,1570.0,B,train
3,U0001,4,272,0,none,1.0,NaN,40.74,310.86,299.83,1558.0,A,train
4,U0001,5,271,0,none,2.0,0.748,38.73,311.32,NaN,1581.0,C,train


Units Dataset:


,unit_id,life_cycles,split,failure_mode
0,U0001,276,train,heat_dissipation
1,U0002,199,train,power
2,U0003,338,train,tool_wear
3,U0004,269,train,power
4,U0005,282,train,heat_dissipation


Ground Truth Dataset:


,unit_id,cycle,damage
0,U0001,1,0.000671
1,U0001,2,0.001653
2,U0001,3,0.002799
3,U0001,4,0.004069
4,U0001,5,0.005439


## 3. Data Understanding and Profiling

### 3.1 Dataset Information

Inspect column names, data types, non-null counts,
and the overall structure of each dataset.

In [10]:
print("Readings Dataset Information:")
readings.info()

print("\nUnits Dataset Information:")
units.info()

print("\nGround Truth Dataset Information:")
ground_truth.info()

Readings Dataset Information:
<class 'pandas.DataFrame'>
RangeIndex: 23118 entries, 0 to 23117
Data columns (total 13 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   unit_id                23118 non-null  str    
 1   cycle                  23118 non-null  int64  
 2   rul_cycles             23118 non-null  int64  
 3   machine_failure        23118 non-null  int64  
 4   failure_mode           23118 non-null  str    
 5   tool_wear_min          23118 non-null  float64
 6   vibration_mm_s         13771 non-null  float64
 7   torque_nm              15617 non-null  float64
 8   process_temperature_k  23118 non-null  float64
 9   air_temperature_k      16848 non-null  float64
 10  rotational_speed_rpm   23118 non-null  float64
 11  control_type           23118 non-null  str    
 12  split                  23118 non-null  str    
dtypes: float64(6), int64(3), str(4)
memory usage: 2.6 MB

Units Dataset Information:
<class

### 3.2 Data Dictionary

Understand the meaning, measurement unit,
and purpose of each column.

| Table | Column | Description | Role |
| :--- | :--- | :--- | :--- |
| readings | unit_id | Machine identifier | Key |
| readings | cycle | Operating cycle | Key / Feature |
| readings | rul_cycles | Remaining useful life | Target |
| readings | machine_failure | Failure indicator | Label |
| readings | failure_mode | Recorded failure type | Label |
| readings | tool_wear_min | Tool wear in minutes | Sensor |
| readings | vibration_mm_s | Vibration in mm/s | Sensor |
| readings | torque_nm | Torque in Nm | Sensor |
| readings | process_temperature_k | Process temperature in Kelvin | Sensor |
| readings | air_temperature_k | Air temperature in Kelvin | Sensor |
| readings | rotational_speed_rpm | Rotational speed in RPM | Sensor |
| readings | control_type | Control configuration | Categorical |
| readings | split | Train/test assignment | Metadata |
| units | unit_id | Machine identifier | Primary key |
| units | life_cycles | Total machine lifetime | Reference |
| units | split | Train/test assignment | Metadata |
| units | failure_mode | Final failure type | Reference |
| ground_truth | unit_id | Machine identifier | Key |
| ground_truth | cycle | Operating cycle | Key |
| ground_truth | damage | Simulated damage value | Ground truth |

### 3.3 Descriptive Statistics

Examine the statistical properties of numerical
and categorical variables to identify unusual values.

In [11]:
print("Readings Statistics:")
display(readings.describe().T)

print("Units Statistics:")
display(units.describe().T)

print("Ground Truth Statistics:")
display(ground_truth.describe().T)

Readings Statistics:


,count,mean,std,min,25%,50%,75%,max
cycle,23118.0,120.828662,74.733522,1.000,58.000,116.000,176.000,358.000
rul_cycles,23118.0,119.828662,74.733522,0.000,57.000,115.000,175.000,357.000
machine_failure,23118.0,0.004326,0.065629,0.000,0.000,0.000,0.000,1.000
tool_wear_min,23118.0,127.554287,93.157870,-2.000,47.000,115.000,194.000,435.000
vibration_mm_s,13771.0,1.947476,1.327383,0.652,0.963,1.399,2.519,9.015
torque_nm,15617.0,47.157019,5.667580,32.760,42.680,46.510,51.020,70.520
process_temperature_k,23118.0,315.410194,3.714894,309.510,312.890,314.550,316.400,328.890
air_temperature_k,16848.0,300.283925,0.452791,298.610,299.970,300.280,300.590,302.130
rotational_speed_rpm,23118.0,1451.124968,101.466330,1044.000,1383.000,1463.000,1531.000,1699.000


Units Statistics:


,count,mean,std,min,25%,50%,75%,max
life_cycles,100.0,231.18,47.043582,128.0,202.0,225.5,269.0,358.0


Ground Truth Statistics:


,count,mean,std,min,25%,50%,75%,max
cycle,23118.0,120.828662,74.733522,1.000000,58.000000,116.000000,176.000000,358.0
damage,23118.0,0.436947,0.298372,0.000479,0.166908,0.408485,0.690614,1.0


### 3.4 Unique Values

Identify the number of distinct values in each
column and examine categorical variables.

In [12]:
print("Readings Unique Values:")
display(readings.nunique())

print("Units Unique Values:")
display(units.nunique())

print("Ground Truth Unique Values:")
display(ground_truth.nunique())

Readings Unique Values:


unit_id                   100
cycle                     358
rul_cycles                358
machine_failure             2
failure_mode                5
tool_wear_min             430
vibration_mm_s           3927
torque_nm                2536
process_temperature_k    1760
air_temperature_k         301
rotational_speed_rpm      590
control_type                3
split                       2
dtype: int64

Units Unique Values:


unit_id         100
life_cycles      73
split             2
failure_mode      4
dtype: int64

Ground Truth Unique Values:


unit_id      100
cycle        358
damage     14756
dtype: int64

In [14]:
categorical_columns = [
    "control_type",
    "failure_mode",
    "machine_failure",
    "split"
]

for col in categorical_columns:
    print(f"\n{col}:")
    print(readings[col].value_counts(dropna=False))


control_type:
control_type
A    9347
B    7501
C    6270
Name: count, dtype: int64

failure_mode:
failure_mode
none                23018
tool_wear              38
heat_dissipation       29
power                  20
overstrain             13
Name: count, dtype: int64

machine_failure:
machine_failure
0    23018
1      100
Name: count, dtype: int64

split:
split
train    18801
test      4317
Name: count, dtype: int64


### 3.5 Missing Values

Calculate the count and percentage of missing
values in each dataset.

In [15]:
def missing_summary(df):
    return pd.DataFrame({
        "Missing Count": df.isnull().sum(),
        "Missing Percentage (%)":
            (df.isnull().mean() * 100).round(2)
    })

In [17]:
print("Readings Missing Values:")
display(missing_summary(readings))

print("Units Missing Values:")
display(missing_summary(units))

print("Ground Truth Missing Values:")
display(missing_summary(ground_truth))

Readings Missing Values:


,Missing Count,Missing Percentage (%)
unit_id,0,0.00
cycle,0,0.00
rul_cycles,0,0.00
machine_failure,0,0.00
failure_mode,0,0.00
tool_wear_min,0,0.00
vibration_mm_s,9347,40.43
torque_nm,7501,32.45
process_temperature_k,0,0.00
air_temperature_k,6270,27.12


Units Missing Values:


,Missing Count,Missing Percentage (%)
unit_id,0,0.0
life_cycles,0,0.0
split,0,0.0
failure_mode,0,0.0


Ground Truth Missing Values:


,Missing Count,Missing Percentage (%)
unit_id,0,0.0
cycle,0,0.0
damage,0,0.0


### 3.6 Duplicate Records

Identify duplicate rows that may affect data
quality and database migration.

In [18]:
duplicate_summary = pd.DataFrame({
    "Dataset": ["readings", "units", "ground_truth"],
    "Duplicate Rows": [
        readings.duplicated().sum(),
        units.duplicated().sum(),
        ground_truth.duplicated().sum()
    ]
})

display(duplicate_summary)

,Dataset,Duplicate Rows
0,readings,0
1,units,0
2,ground_truth,0


## 4. Initial Findings

### Dataset Structure

- The dataset contains 46,336 records across three
  related tables, with 20 columns in total.
- The dataset includes 100 unique machines.
- Machine lifetimes range from 128 to 358 cycles,
  with an average of 231.18 cycles.
- The dataset dimensions match the project requirements.

### Data Quality Findings

- Missing values are present in three sensor columns:
  - Vibration: 9,347 missing values (40.43%).
  - Torque: 7,501 missing values (32.45%).
  - Air temperature: 6,270 missing values (27.12%).

- No missing values were detected in the units
  and ground truth tables.

- No complete duplicate records were detected
  across the three datasets.

- Negative values were detected in tool_wear_min,
  with a minimum value of -2.0 minutes.
  These values require further investigation.

### Preliminary Missingness Findings

- The missing-value counts match the corresponding
  control-type frequencies.

- This suggests a possible structural missingness
  pattern related to control_type.

- Further analysis is required to confirm the
  missingness mechanism and determine appropriate
  treatment strategies.

### Failure Distribution

- The readings table contains 23,018 normal records
  and 100 failure records.

- Four failure modes were identified:
  tool wear, heat dissipation, power, and overstrain.

### Conclusion

The raw datasets were successfully loaded and profiled
without modifying the original data.

The next notebook will focus on data integrity,
cross-table consistency, invalid sensor values,
and missingness mechanisms before recommending
appropriate data cleaning strategies.